### File ingestion

In [0]:
#Initializing project folders for raw files

# dbutils.fs.mkdirs("/Volumes/workspace/bronze/raw/defects_project/excel/")
# dbutils.fs.mkdirs("/Volumes/workspace/bronze/raw/defects_project/csv/")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bronze/raw/defects_project/csv/"))

In [0]:
pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

ladp_excel_path= "/Volumes/workspace/bronze/raw/defects_project/excel/LDAP_GROUP_USERS_FOR ALL APPLICATION LEVEL.xlsx"
#audit_excel_path= "/Volumes/workspace/bronze/raw/defects_project/excel/audit log - extract.xlsx"

ladp_csv_path = "/Volumes/workspace/bronze/raw/defects_project/csv/ladp.csv"
audit_csv_path = "/Volumes/workspace/bronze/raw/defects_project/csv/audit_mar.csv"

#Reading excel files
ladp_df = pd.read_excel(ladp_excel_path,sheet_name="DEFECT ANALYSES USERS")
#audit_df = pd.read_excel(audit_excel_path,sheet_name="Query 1")

ladp_df.to_csv(ladp_csv_path,index = False)
#audit_df.to_csv(audit_csv_path,index = False)

print("CSV files created succesfully")
print("ladp_csv",ladp_csv_path)
print("audit_mar_csv",audit_csv_path)


In [0]:
import pandas as pd

user_details_excel = "/Volumes/workspace/bronze/raw/defects_project/excel/ALL ENTITY USER DETAILS.xlsx"

user_details_csv = "/Volumes/workspace/bronze/raw/defects_project/csv/user_details.csv"


# xls = pd.ExcelFile(user_details_excel)
# print(xls.sheet_names)
user_detailss_df = pd.read_excel(user_details_excel,sheet_name='All User Details ')

user_detailss_df.to_csv(user_details_csv,index = False)

print("CSV files created succesfully")
print("user_details_csv",user_details_csv)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bronze/raw/defects_project/csv/"))

### Bronze Delta tables

LADP table

In [0]:
%sql
Create or replace table workspace.bronze.ldap_users_bronze
using delta 
as
select * from 
read_files("/Volumes/workspace/bronze/raw/defects_project/csv/ladp.csv",
format => "csv",
inferSchema => "true",
header => "true")

In [0]:
%sql
select * from workspace.bronze.ldap_users_bronze

User details table

In [0]:
import re
from pyspark.sql.functions import col

# Step 1: Read the CSV
user_details_df = spark.read.csv(
    "/Volumes/workspace/bronze/raw/defects_project/csv/user_details.csv",
    header=True,
    inferSchema=True
)

# Step 2: Capture the first row as the real header
new_header = user_details_df.first()

# Step 3: Remove that header row from the data
user_details_df = user_details_df.filter(
    col("All Entity User Details – Lab Wise") != "SL NO"
)

# Step 4: Apply the first row values as actual column names
user_details_df = user_details_df.toDF(*list(new_header))

# Step 5: Clean column names
def clean_col(col_name):
    col_name = col_name.lower().strip()
    col_name = re.sub(r"[^\w]", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name)
    return col_name.strip("_")

user_details_df = user_details_df.toDF(*[clean_col(c) for c in user_details_df.columns])

# Step 6: Write to Bronze Delta table
user_details_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.user_details_bronze")

print("Bronze table created successfully")
print(user_details_df.columns)

Audit_log table

In [0]:
%sql
drop table if exists workspace.bronze.audit_bronze

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
import re

audit_df = spark.read.csv(
    "/Volumes/workspace/bronze/raw/defects_project/csv/audit_*.csv",
    header=True,
    inferSchema=False  # Read everything as string
)

def clean_col(col):
    col = col.lower().strip()
    col = re.sub(r"[^\w]", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")

df = audit_df.toDF(*[clean_col(c) for c in audit_df.columns])

df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.bronze.audit_bronze"
)

Check

In [0]:
%sql
-- show tables in workspace.bronze

In [0]:
%sql
-- Select * from workspace.bronze.audit_bronze
-- limit 10;


In [0]:
# %sql
# select * from workspace.bronze.audit_bronze

In [0]:
%sql
-- show tables in workspace.bronze

In [0]:
# df = spark.read.table("workspace.bronze.audit_bronze")

# print(df.columns)

In [0]:
# df = spark.read.table("workspace.bronze.ladp_users_bronze")

# print(df.columns)

In [0]:
%sql
-- SELECT COUNT(*) FROM workspace.bronze.audit_bronze
-- WHERE start_datetime LIKE '2026/04/%';

-- SELECT COUNT(*) FROM workspace.bronze.audit_bronze

SELECT 
    CASE
        WHEN start_datetime LIKE '2026/01/%' THEN 'Jan'
        WHEN start_datetime LIKE '2026/02/%' THEN 'Feb'
        WHEN start_datetime LIKE '2026/03/%' THEN 'Mar'
        WHEN start_datetime LIKE '2026/04/%' THEN 'Apr'
        WHEN start_datetime LIKE '2026/05/%' THEN 'May'
        WHEN start_datetime LIKE '2026-%' THEN 'dash_format'
        ELSE 'Other'
    END AS month,
    COUNT(*) AS cnt
FROM workspace.bronze.audit_bronze
GROUP BY 1
ORDER BY 1

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bronze/raw/defects_project/csv/"))